# Modelo de temperatura — Ley de enfriamiento de Newton

Elaborado por: *(nombre del estudiante)*

**Materia:** Ecuaciones Diferenciales

Este cuaderno modela cómo cambia la temperatura de un objeto que está en contacto con un ambiente a temperatura $T_a$.

$$\frac{dT}{dt} = -k\,(T - T_a), \qquad T(0) = T_0$$

- $T(t)$: temperatura del objeto en el tiempo $t$ (°C)
- $T_a$: temperatura del ambiente (°C)
- $k > 0$: constante de transferencia de calor (1/min)
- $T_0$: temperatura inicial del objeto (°C)

**Contenido**
1. Imports
2. Clase `Escenario` (EDO, solución exacta, Euler, equilibrio)
3. Punto de equilibrio y estabilidad
4. Casos de estudio y gráficas de series de tiempo
5. Campo de direcciones y línea de fase
6. Método de Euler vs. solución exacta (y estabilidad numérica)
7. Modelo modificado 1: temperatura ambiente variable
8. Modelo modificado 2: sistema acoplado objeto–ambiente

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp

plt.rcParams["figure.figsize"] = (10, 4)

## 2. Clase `Escenario`

Agrupa los parámetros del modelo y los métodos para simular, calcular el equilibrio y graficar.

In [ ]:
class Escenario:
    """Ley de enfriamiento de Newton: dT/dt = -k (T - T_amb)."""

    def __init__(self, k=0.15, T_amb=20.0):
        # Parámetros del sistema
        self.k = k
        self.T_amb = T_amb

        # Símbolos algebraicos
        self.t_symb, self.T_symb = sp.symbols("t T", real=True)

        # Ecuación del sistema
        self.ecuacion = -self.k * (self.T_symb - self.T_amb)

    # ---------- EDO y soluciones ----------
    def edo(self, tiempo, T):
        return -self.k * (T - self.T_amb)

    def solucion_exacta(self, T0, t):
        """T(t) = T_amb + (T0 - T_amb) e^(-k t)"""
        return self.T_amb + (T0 - self.T_amb) * np.exp(-self.k * t)

    def solucion_simbolica(self, T0):
        """Resuelve la EDO con sympy (dsolve) para mostrar el procedimiento analítico."""
        T = sp.Function("T")
        eq = sp.Eq(T(self.t_symb).diff(self.t_symb),
                   -sp.nsimplify(self.k) * (T(self.t_symb) - sp.nsimplify(self.T_amb)))
        return sp.dsolve(eq, T(self.t_symb), ics={T(0): sp.nsimplify(T0)})

    def simular(self, T0, tiempo_final=30, paso=0.01):
        """Integra numéricamente con RK45 (scipy)."""
        self.T0 = T0
        self.tiempo_final = tiempo_final
        self.paso = paso

        num_puntos = int(tiempo_final / paso) + 1
        self.t_eval = np.linspace(0, tiempo_final, num_puntos)

        solucion = solve_ivp(self.edo, [0, tiempo_final], [T0],
                             t_eval=self.t_eval, method="RK45")
        self.tiempo, self.temperatura = solucion.t, solucion.y[0]
        return self.tiempo, self.temperatura

    def euler(self, T0, tiempo_final=30, h=0.5):
        """Método de Euler explícito: T_{n+1} = T_n + h f(t_n, T_n)."""
        n = int(round(tiempo_final / h))
        t = np.linspace(0, n * h, n + 1)
        T = np.zeros(n + 1)
        T[0] = T0
        for i in range(n):
            T[i + 1] = T[i] + h * self.edo(t[i], T[i])
        return t, T

    # ---------- Equilibrio y estabilidad ----------
    def calcular_puntos_equilibrio(self):
        puntos = sp.solve(self.ecuacion, self.T_symb, dict=True)
        jacobiano = sp.Matrix([self.ecuacion]).jacobian([self.T_symb])

        resultados = []
        for p in puntos:
            T_eq = float(p[self.T_symb])
            J = jacobiano.subs({self.T_symb: T_eq})
            autovalores = {round(float(ev), 2): mult for ev, mult in J.eigenvals().items()}
            lam = list(autovalores.keys())[0]
            if lam < 0:
                estabilidad = "asintóticamente estable (atractor)"
            elif lam > 0:
                estabilidad = "inestable (repulsor)"
            else:
                estabilidad = "no concluyente"
            resultados.append({
                "punto_equilibrio": round(T_eq, 2),
                "jacobiano": [[round(float(J[0, 0]), 2)]],
                "autovalores": autovalores,
                "estabilidad": estabilidad,
            })
        return resultados

    def tiempo_para_alcanzar(self, T0, T_objetivo):
        """Tiempo en que el objeto llega a T_objetivo (despeje de la solución exacta)."""
        razon = (T_objetivo - self.T_amb) / (T0 - self.T_amb)
        if razon <= 0 or razon >= 1:
            return None  # nunca la alcanza
        return -np.log(razon) / self.k

    # ---------- Gráficas ----------
    def graficar_series(self, ax=None, mostrar=True):
        if ax is None:
            plt.figure(figsize=(10, 4))
            ax = plt.gca()
        ax.plot(self.tiempo, self.temperatura, color="tab:red", label=f"T(t), T0 = {self.T0}")
        ax.axhline(self.T_amb, color="gray", linestyle="--", label=f"T ambiente = {self.T_amb}")
        ax.set_xlabel("Tiempo (min)")
        ax.set_ylabel("Temperatura (°C)")
        ax.set_title(f"Evolución temporal de la temperatura (k = {self.k})")
        ax.legend()
        ax.grid(True)
        if mostrar:
            plt.show()

    def campo_de_direcciones(self, ax, t_max=30, T_min=0, T_max=100, densidad=20):
        """Campo de direcciones en el plano (t, T): cada flecha tiene pendiente dT/dt."""
        t_g, T_g = np.meshgrid(np.linspace(0, t_max, densidad),
                               np.linspace(T_min, T_max, densidad))
        pend = self.edo(t_g, T_g)
        # Se normalizan las flechas respetando la escala de los ejes
        escala_x, escala_y = t_max, (T_max - T_min)
        u = np.ones_like(pend) / escala_x
        v = pend / escala_y
        norma = np.hypot(u, v)
        ax.quiver(t_g, T_g, u / norma, v / norma, angles="xy", color="gray", alpha=0.6)
        ax.set_xlim(0, t_max)
        ax.set_ylim(T_min, T_max)

    def linea_de_fase(self, ax, T_min=0, T_max=100):
        """Línea de fase: dT/dt en función de T. Los cortes con 0 son equilibrios."""
        T = np.linspace(T_min, T_max, 200)
        ax.plot(T, self.edo(0, T), color="tab:blue")
        ax.axhline(0, color="black", linewidth=0.8)
        ax.plot(self.T_amb, 0, "o", color="tab:red", label=f"Equilibrio T = {self.T_amb}")
        ax.set_xlabel("T (°C)")
        ax.set_ylabel("dT/dt")
        ax.set_title("Línea de fase")
        ax.legend()
        ax.grid(True)

## 3. Punto de equilibrio y estabilidad

Se busca $f(T)=0$:

$$-k(T - T_a) = 0 \;\Rightarrow\; T^* = T_a$$

El jacobiano de una ecuación escalar es $f'(T) = -k$, así que el único autovalor es $\lambda = -k < 0$ para todo $k>0$.
Por tanto, $T^* = T_a$ es **asintóticamente estable**: sin importar la temperatura inicial, el objeto tiende a la temperatura del ambiente.

In [ ]:
modelo = Escenario(k=0.15, T_amb=20.0)
modelo.simular(T0=90.0, tiempo_final=30, paso=0.01)

equilibrios = modelo.calcular_puntos_equilibrio()

print("Puntos de equilibrio y sus autovalores:")
for eq in equilibrios:
    print(f"Punto de equilibrio: {eq['punto_equilibrio']}")
    print(f"Jacobiano: {eq['jacobiano']}")
    print(f"Autovalores: {eq['autovalores']}")
    print(f"Estabilidad: {eq['estabilidad']}\n")

**Solución analítica (sympy)** — separación de variables / factor integrante:

$$T(t) = T_a + (T_0 - T_a)\,e^{-kt}$$

In [ ]:
modelo.solucion_simbolica(T0=90)

In [ ]:
t_obj = modelo.tiempo_para_alcanzar(T0=90, T_objetivo=40)
print(f"El objeto pasa de 90 °C a 40 °C en {t_obj:.2f} min")
print(f"Vida media térmica (tiempo para cerrar la mitad de la brecha): {np.log(2)/modelo.k:.2f} min")

## 4. Casos de estudio

Se usa el mismo ambiente ($T_a = 20$ °C, $k = 0.15$) con diferentes temperaturas iniciales, y un caso con otro $k$.

| Caso | $T_0$ | $T_a$ | $k$ | Descripción |
|---|---|---|---|---|
| 1 | 90 | 20 | 0.15 | Café caliente que se enfría |
| 2 | 5 | 20 | 0.15 | Bebida fría que se calienta |
| 3 | 20 | 20 | 0.15 | Objeto ya en equilibrio |
| 4 | 90 | 20 | 0.6 | Mismo café con mejor transferencia de calor (k alto) |

In [ ]:
caso1 = Escenario(k=0.15, T_amb=20.0)
caso1.simular(T0=90.0, tiempo_final=30, paso=0.01)

caso2 = Escenario(k=0.15, T_amb=20.0)
caso2.simular(T0=5.0, tiempo_final=30, paso=0.01)

caso3 = Escenario(k=0.15, T_amb=20.0)
caso3.simular(T0=20.0, tiempo_final=30, paso=0.01)

caso4 = Escenario(k=0.6, T_amb=20.0)
caso4.simular(T0=90.0, tiempo_final=30, paso=0.01)

In [ ]:
caso1.graficar_series()

In [ ]:
caso2.graficar_series()

In [ ]:
caso3.graficar_series()

In [ ]:
caso4.graficar_series()

### Comparación de todos los casos

In [ ]:
plt.figure(figsize=(10, 5))
for nombre, caso in [("Caso 1 (T0=90, k=0.15)", caso1), ("Caso 2 (T0=5, k=0.15)", caso2),
                     ("Caso 3 (T0=20, k=0.15)", caso3), ("Caso 4 (T0=90, k=0.6)", caso4)]:
    plt.plot(caso.tiempo, caso.temperatura, label=nombre)
plt.axhline(20, color="gray", linestyle="--", label="T ambiente")
plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Todas las trayectorias convergen a T ambiente")
plt.legend()
plt.grid(True)
plt.savefig("resources/comparacion_casos.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Campo de direcciones y línea de fase

El campo de direcciones muestra la pendiente $dT/dt$ en cada punto $(t, T)$. Las trayectorias son curvas tangentes a las flechas.
Para una EDO escalar el "retrato de fase" es la **línea de fase**: la gráfica de $dT/dt$ contra $T$.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Campo de direcciones + trayectorias
caso1.campo_de_direcciones(axs[0], t_max=30, T_min=0, T_max=100)
for nombre, caso in [("T0 = 90", caso1), ("T0 = 5", caso2), ("T0 = 20", caso3)]:
    axs[0].plot(caso.tiempo, caso.temperatura, linewidth=2, label=nombre)
axs[0].set_xlabel("Tiempo (min)")
axs[0].set_ylabel("Temperatura (°C)")
axs[0].set_title("Campo de direcciones y trayectorias")
axs[0].legend()
axs[0].grid(True)

# Línea de fase
caso1.linea_de_fase(axs[1], T_min=0, T_max=100)

plt.tight_layout()
plt.savefig("resources/campo_direcciones.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Método de Euler vs. solución exacta

Se compara Euler con distintos tamaños de paso $h$ frente a la solución exacta.

In [ ]:
T0 = 90.0
modelo = Escenario(k=0.15, T_amb=20.0)
t_fino = np.linspace(0, 30, 1000)

plt.figure(figsize=(10, 5))
plt.plot(t_fino, modelo.solucion_exacta(T0, t_fino), "k", linewidth=2, label="Solución exacta")
for h in [4.0, 2.0, 1.0, 0.25]:
    t_e, T_e = modelo.euler(T0, tiempo_final=30, h=h)
    plt.plot(t_e, T_e, "o-", markersize=4, label=f"Euler h = {h}")
plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Método de Euler vs. solución exacta")
plt.legend()
plt.grid(True)
plt.savefig("resources/euler_vs_exacta.png", dpi=150, bbox_inches="tight")
plt.show()

### Error de Euler en función de $h$

In [ ]:
print(f"{'h':>6} | {'Error máx (°C)':>15}")
print("-" * 25)
errores = []
hs = [4.0, 2.0, 1.0, 0.5, 0.25, 0.125, 0.0625]
for h in hs:
    t_e, T_e = modelo.euler(T0, tiempo_final=32, h=h)
    err = np.max(np.abs(T_e - modelo.solucion_exacta(T0, t_e)))
    errores.append(err)
    print(f"{h:>6} | {err:>15.5f}")

plt.figure(figsize=(7, 4))
plt.loglog(hs, errores, "o-")
plt.xlabel("Paso h")
plt.ylabel("Error máximo (°C)")
plt.title("El error de Euler decrece linealmente con h (orden 1)")
plt.grid(True, which="both")
plt.show()

### Estabilidad numérica de Euler

Para esta EDO, Euler da $T_{n+1} - T_a = (1 - hk)(T_n - T_a)$. Por eso:

- Si $hk < 1$: la solución numérica decrece monótonamente (comportamiento correcto).
- Si $1 < hk < 2$: oscila pero converge (comportamiento incorrecto, aunque estable).
- Si $hk > 2$: **diverge** aunque la solución real converge.

Con $k = 0.15$ el límite de estabilidad es $h = 2/k \approx 13.3$. Para verlo con pasos más pequeños, se usa $k = 0.6$ (límite $h \approx 3.33$).

In [ ]:
modelo_rapido = Escenario(k=0.6, T_amb=20.0)
t_fino = np.linspace(0, 20, 1000)

plt.figure(figsize=(10, 5))
plt.plot(t_fino, modelo_rapido.solucion_exacta(90, t_fino), "k", linewidth=2, label="Exacta")
for h, estilo in [(1.0, "hk = 0.6 (monótona)"), (2.5, "hk = 1.5 (oscila y converge)"), (3.6, "hk = 2.16 (diverge)")]:
    t_e, T_e = modelo_rapido.euler(90, tiempo_final=20, h=h)
    plt.plot(t_e, T_e, "o-", markersize=4, label=f"h = {h}: {estilo}")
plt.ylim(-100, 200)
plt.xlabel("Tiempo (min)")
plt.ylabel("Temperatura (°C)")
plt.title("Estabilidad del método de Euler (k = 0.6)")
plt.legend()
plt.grid(True)
plt.savefig("resources/estabilidad_euler.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Modelo modificado 1: temperatura ambiente variable

En la vida real el ambiente no es constante. Se modela el ciclo día–noche con una senoidal:

$$\frac{dT}{dt} = -k\,\bigl(T - T_a(t)\bigr), \qquad T_a(t) = T_m + A\sin\!\left(\frac{2\pi t}{P}\right)$$

- $T_m$: temperatura media del ambiente
- $A$: amplitud de la oscilación
- $P$: periodo (24 h)

Ahora **no existe un punto de equilibrio fijo**: el objeto sigue al ambiente con un retraso (desfase) y menor amplitud.

In [ ]:
class EscenarioVariable(Escenario):
    """dT/dt = -k (T - T_amb(t)), con T_amb(t) = T_m + A sin(2 pi t / P)."""

    def __init__(self, k=0.5, T_m=20.0, A=8.0, P=24.0):
        super().__init__(k=k, T_amb=T_m)
        self.T_m, self.A, self.P = T_m, A, P

    def T_ambiente(self, t):
        return self.T_m + self.A * np.sin(2 * np.pi * t / self.P)

    def edo(self, tiempo, T):
        return -self.k * (T - self.T_ambiente(tiempo))

    def graficar_series(self, ax=None, mostrar=True):
        plt.figure(figsize=(10, 4))
        plt.plot(self.tiempo, self.temperatura, color="tab:red", label=f"Objeto (T0 = {self.T0})")
        plt.plot(self.tiempo, self.T_ambiente(self.tiempo), "--", color="gray", label="Ambiente T_a(t)")
        plt.xlabel("Tiempo (h)")
        plt.ylabel("Temperatura (°C)")
        plt.title("Objeto en un ambiente con ciclo día–noche")
        plt.legend()
        plt.grid(True)
        plt.show()

In [ ]:
variable = EscenarioVariable(k=0.5, T_m=20.0, A=8.0, P=24.0)
variable.simular(T0=35.0, tiempo_final=72, paso=0.01)
variable.graficar_series()

**Amplitud y desfase en régimen permanente.** Para $T_a(t)=T_m + A\sin(\omega t)$, la solución estacionaria es
$T_{ss}(t)=T_m + \dfrac{A}{\sqrt{1+(\omega/k)^2}}\sin(\omega t - \varphi)$, con $\varphi = \arctan(\omega/k)$.

In [ ]:
omega = 2 * np.pi / variable.P
amp_teorica = variable.A / np.sqrt(1 + (omega / variable.k) ** 2)
desfase_horas = np.arctan(omega / variable.k) / omega

# Amplitud numérica: último periodo de la simulación
ult = variable.tiempo > variable.tiempo_final - variable.P
amp_numerica = (variable.temperatura[ult].max() - variable.temperatura[ult].min()) / 2

print(f"Amplitud teórica del objeto:  {amp_teorica:.3f} °C")
print(f"Amplitud numérica del objeto: {amp_numerica:.3f} °C")
print(f"Retraso respecto al ambiente: {desfase_horas:.2f} h")

## 8. Modelo modificado 2: sistema acoplado objeto–ambiente

Si el ambiente (por ejemplo, una habitación pequeña) también se calienta con el objeto, ya no se puede suponer $T_a$ constante.
Se tiene un **sistema de dos ecuaciones**, parecido al de presa–depredador:

$$\frac{dT}{dt} = -k_1\,(T - T_a), \qquad \frac{dT_a}{dt} = k_2\,(T - T_a) - k_3\,(T_a - T_{ext})$$

- $T$: temperatura del objeto, $T_a$: temperatura del aire de la habitación
- $k_1$: transferencia objeto → aire, $k_2$: calentamiento del aire por el objeto
- $k_3$: pérdida de calor del aire hacia el exterior (a temperatura $T_{ext}$ constante)

In [ ]:
class EscenarioAcoplado:
    def __init__(self, k1=0.3, k2=0.1, k3=0.05, T_ext=15.0):
        self.k1, self.k2, self.k3, self.T_ext = k1, k2, k3, T_ext
        self.T_symb, self.Ta_symb = sp.symbols("T T_a", real=True)
        self.f1 = -self.k1 * (self.T_symb - self.Ta_symb)
        self.f2 = self.k2 * (self.T_symb - self.Ta_symb) - self.k3 * (self.Ta_symb - self.T_ext)

    def sistema_edo(self, tiempo, estado):
        T, Ta = estado
        return [-self.k1 * (T - Ta),
                self.k2 * (T - Ta) - self.k3 * (Ta - self.T_ext)]

    def simular(self, T0, Ta0, tiempo_final=80, paso=0.01):
        self.T0, self.Ta0 = T0, Ta0
        t_eval = np.linspace(0, tiempo_final, int(tiempo_final / paso) + 1)
        sol = solve_ivp(self.sistema_edo, [0, tiempo_final], [T0, Ta0], t_eval=t_eval, method="RK45")
        self.tiempo, self.temperatura, self.temp_aire = sol.t, sol.y[0], sol.y[1]
        return self.tiempo, self.temperatura, self.temp_aire

    def calcular_puntos_equilibrio(self):
        puntos = sp.solve([self.f1, self.f2], (self.T_symb, self.Ta_symb), dict=True)
        J = sp.Matrix([self.f1, self.f2]).jacobian([self.T_symb, self.Ta_symb])
        resultados = []
        for p in puntos:
            T_eq, Ta_eq = float(p[self.T_symb]), float(p[self.Ta_symb])
            autoval = {complex(round(float(sp.re(ev)), 3), round(float(sp.im(ev)), 3)): m
                       for ev, m in J.eigenvals().items()}
            resultados.append({"punto_equilibrio": (round(T_eq, 2), round(Ta_eq, 2)),
                               "jacobiano": [[round(float(J[i, j]), 3) for j in range(2)] for i in range(2)],
                               "autovalores": autoval})
        return resultados

    def graficar_series(self):
        plt.figure(figsize=(10, 4))
        plt.plot(self.tiempo, self.temperatura, color="tab:red", label=f"Objeto (T0 = {self.T0})")
        plt.plot(self.tiempo, self.temp_aire, color="tab:blue", label=f"Aire (Ta0 = {self.Ta0})")
        plt.axhline(self.T_ext, color="gray", linestyle="--", label=f"Exterior = {self.T_ext}")
        plt.xlabel("Tiempo (min)")
        plt.ylabel("Temperatura (°C)")
        plt.title("Sistema acoplado objeto–aire–exterior")
        plt.legend()
        plt.grid(True)
        plt.show()

    def retrato_de_fase(self, ax=None, paso=400):
        if ax is None:
            ax = plt.gca()
        ax.plot(self.temperatura, self.temp_aire, label=f"T0 = {self.T0}, Ta0 = {self.Ta0}")
        # Flechas con la dirección del campo vectorial en puntos de la trayectoria
        xs = self.temperatura[0:-1:paso]
        ys = self.temp_aire[0:-1:paso]
        dxs = -self.k1 * (xs - ys)
        dys = self.k2 * (xs - ys) - self.k3 * (ys - self.T_ext)
        norma = np.hypot(dxs, dys) + 1e-12
        ax.quiver(xs, ys, dxs / norma, dys / norma, angles="xy", width=0.004, zorder=0)

In [ ]:
acoplado = EscenarioAcoplado(k1=0.3, k2=0.1, k3=0.05, T_ext=15.0)
acoplado.simular(T0=90.0, Ta0=20.0, tiempo_final=80)

print("Puntos de equilibrio y sus autovalores:")
for eq in acoplado.calcular_puntos_equilibrio():
    print(f"Punto de equilibrio: {eq['punto_equilibrio']}")
    print(f"Matriz Jacobiana: {eq['jacobiano']}")
    print(f"Autovalores: {eq['autovalores']}\n")

acoplado.graficar_series()

Los dos autovalores son reales y negativos, así que el equilibrio $(T_{ext}, T_{ext})$ es un **nodo estable**: objeto y aire terminan igualando la temperatura exterior.
Nótese que, a diferencia del modelo presa–depredador, **no hay oscilaciones** (no hay autovalores complejos).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for T0, Ta0 in [(90, 20), (5, 20), (60, 5), (10, 40)]:
    caso = EscenarioAcoplado(k1=0.3, k2=0.1, k3=0.05, T_ext=15.0)
    caso.simular(T0=T0, Ta0=Ta0, tiempo_final=80)
    caso.retrato_de_fase(ax)
ax.plot(15, 15, "ko", label="Equilibrio (15, 15)")
ax.set_xlabel("Temperatura del objeto T")
ax.set_ylabel("Temperatura del aire T_a")
ax.set_title("Retrato de fase del sistema acoplado")
ax.legend()
ax.grid(True)
plt.savefig("resources/retrato_fase_acoplado.png", dpi=150, bbox_inches="tight")
plt.show()

## Conclusiones

- La EDO $dT/dt = -k(T-T_a)$ tiene un único equilibrio $T^*=T_a$, asintóticamente estable ($\lambda=-k$).
- La solución exacta $T_a + (T_0-T_a)e^{-kt}$ coincide con RK45; el método de Euler converge con orden 1 y es estable solo si $hk<2$.
- Con ambiente variable, el objeto sigue al ambiente con menor amplitud y un retraso temporal.
- Con ambiente acoplado, el sistema 2×2 converge a un nodo estable sin oscilaciones.